## DestinE Data Streaming

This service offers compressed climate and era5 data and makes it available via a high quality and memory efficient streaming solution. The [SSIM](https://en.wikipedia.org/wiki/Structural_similarity_index_measure) and the mean relative error serve as quality measures.

<div style='white-space: nowrap', align='center'>

<div style='display:inline-block', align='center'>Era5 2 meter dewpoint temperature (01-01-1940 09:00)<br>
<img src="../images/2d9_og_.jpeg" width="450px"><br><img src="../images/2d9_cp_.jpeg" width="450px"><br>Mean SSIM: 0.996<br>Compression rate 1:13<br>Mean relative error 0.1 %</div>

<div style='display:inline-block', align='center'>Era 5 10 metre U wind component (01-01-1940 09:00)<br>
<img src="../images/10u9_og_.jpeg" width="450px"><br><img src="../images/10u9_cp_.jpeg" width="450px"><br>Mean SSIM: 0.995<br>Compression rate 1:27<br>Mean relative error 0.3 %</div>

</div>


## Prerequisites
### DestinE Platform Credentials

You need to have an account on the [Destination Earth Platform](https://auth.destine.eu/realms/desp/account).

#### ⚠️ Warning: Authorized Access Only
The usage of this notebook and data access is reserved only to authorized user groups.

## Access the Data
With a DESP account you can access the stream data proposed in this tutorial.

# WIND MAGNITUDES ON THE COASTS OF FRANCE AND GREAT BRITAIN
In this example we are loading east and north facing winds to calculate the total magnitude and will be plotting it over the course of two years, 1941 and 2021.

Both years are sliced directly out of the Zarr-backed stream for our west-coast AOI, so only the chunks covering that region and time span are ever fetched — no per-frame ffmpeg decoding involved.

In [ ]:
%%capture cap
%run ./auth.py

In [ ]:
output = [line for line in cap.stdout.split('\n') if line.strip()]
refresh_token = output[-2]
token = output[-1]

# Imports and general definitions
We start by importing necessary packages. Data access now goes through `dte_zarr`, which exchanges the DESP token for a stream, then a dataset-scoped Zarr token, and opens the result as a lazily-loaded, geo-referenced `xarray.DataArray` backed by `dask` — no ffmpeg decoding involved.

Note: The API token must be set here including the user group. This happens in **Authentification**.

In [ ]:
from dte_zarr import open_stream
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from IPython import display

FORMAT = '%Y-%m-%dT%H:%M'

import time
startt = time.time()


# Parameters for stream access

Here the parameters are set to access the data from the service.

*program_subset*: ERA5 has data from 1940 to 2023 </br>
*variable*: 10u is the horizontal 10m wind towards the East. 10v is the horizontal wind towards the north.</br>
*start_date/end_date* per year: 00:00 on the first day to 23:00 on the last day of 1941 or 2021</br>

In [ ]:
program_subset = "ERA5"
u_wind_variable = "10u"
v_wind_variable = "10v"

years = [1941, 2021]
dates = [
    [
        datetime.strptime(f"{year}-01-01T00:00", FORMAT),
        datetime.strptime(f"{year}-12-31T23:00", FORMAT),
    ]
    for year in years
]

# Both streams are opened once here as lazy, geo-referenced arrays covering
# the full ERA5 period. Per-year and per-AOI slicing happens in the next cells,
# so this is also where all the API calls for this notebook are made.
u_wind = open_stream(token, program_subset, u_wind_variable)
v_wind = open_stream(token, program_subset, v_wind_variable)

# Data specific variables

As an area of interest (AOI), parts of the Atlantic Ocean, the british west coast and french west coast are chosen — now as a real latitude/longitude bounding box instead of a hardcoded pixel slice. Plots will be made for two years. We set some colors to distinguish them and their fits.

In [ ]:
# European west coast (France & Great Britain), as a lat/lon bounding box
aoi_lat = slice(65.0, 42.5)
aoi_lon = slice(-22.5, -0.25)

# Colors for the plot of the data and the fit curves
data_colors = ['#b4e7ff', '#ffc6c6']
fit_colors = ['#6157cc', '#bf4f4f']

# Working with the data

For each year, the u/v wind streams are sliced down to the AOI and the year's time span, the magnitude is computed with Pythagoras' theorem, and averaged spatially over the AOI — all lazily, then triggered with a single `.compute()` per year. `dask` fetches only the chunks of the Zarr store that actually cover the AOI and that year, over HTTP, instead of decoding and iterating through every one of the ~8760 hourly frames one by one.

A polynomial fit is then found through each year's data, to show a trend through the heavily fluctuating measurement data.

In [ ]:
# variable for data storage
magnitudes = [[] for _ in years]
t = []

# variables for the plotted figure
ticks = []
tick_labels = []
y_min = 1.0*10**10
y_max = 0.

# initialize the plot figure
hdisplay = display.display('', display_id=True)
fig, ax = plt.subplots(1, 1, figsize=(15, 6))

ax.set_ylabel(f'Wind Speed in m/s')
plt.title(f'Wind Speeds at Western European Coasts')

# For loop to evaluate the data of all specified years
for i, year in enumerate(years):
    start_date, end_date = dates[i]

    # slice both wind components down to the AOI and this year, then
    # compute the magnitude and its spatial average over the AOI in one go
    u_year = u_wind.sel(time=slice(start_date, end_date), latitude=aoi_lat, longitude=aoi_lon)
    v_year = v_wind.sel(time=slice(start_date, end_date), latitude=aoi_lat, longitude=aoi_lon)
    wind_magnitude = ((u_year**2 + v_year**2)**(1/2)).mean(dim=["latitude", "longitude"]).compute()

    magnitudes[i] = list(wind_magnitude.values)
    time_stamps = pd.to_datetime(wind_magnitude["time"].values)

    if i == 0:
        # adjust the x-axis of the plot to the data, using timestamps
        t = [ts.timestamp() for ts in time_stamps]
        ax.set_xlim(dates[0][0].timestamp(), dates[0][1].timestamp())

        # create time axis labels at the start of each month
        ticks = [ts.timestamp() for ts in time_stamps if ts.hour == 0 and ts.day == 1]
        tick_labels = [ts.strftime('  %B') for ts in time_stamps if ts.hour == 0 and ts.day == 1]

    # scale the data axis to the data seen so far
    y_min = min([y_min, *magnitudes[i]])
    y_max = max([y_max, *magnitudes[i]])
    ax.set_ylim(y_min-1, y_max+1)

    # plot the data curve for this year
    line, = ax.plot(t, magnitudes[i], color=data_colors[i])
    line.set_label(f'{year} data')
    line.set_zorder(i)

    ax.set_xticks(ticks, tick_labels, ha='left')
    ax.grid(True, axis='x')
    hdisplay.update(fig)

    # make a polynomial fit for the current data and plot it
    fit = np.polyfit(t, magnitudes[i], 4)
    fit = np.polyval(fit, t)
    line, = ax.plot(t, fit, color=fit_colors[i])
    line.set_zorder(i + len(years))

    # set a label for the plot and update the figure
    line.set_label(f'{year} fit')
    ax.legend()
    hdisplay.update(fig)

plt.close(fig)
print(f'duration: {time.time()-startt}')